In [4]:
import pandas as pd
import os

def remove_spaces(raw_keywords):
    """
    한 행의 키워드 데이터를 받아 쉼표 기준으로 분리한 뒤,
    각 키워드 내부의 공백(띄어쓰기)을 모두 제거하여 덩어리 키워드로 만듭니다.
    """
    # 값이 비어있을 경우(NaN) 그대로 반환
    if pd.isna(raw_keywords):
        return raw_keywords
    
    final_words = []
    # 1. 쉼표(,)로 분리
    comma_split = str(raw_keywords).split(',')
    
    for item in comma_split:
        # 2. 키워드 내의 모든 공백(띄어쓰기) 제거
        # 예: '인테리어 소품' -> '인테리어소품'
        word_no_space = item.replace(" ", "")
        
        # 공백만 있던 문자열이 아니었다면 리스트에 추가
        if word_no_space: 
            final_words.append(word_no_space)
            
    # 분리된 단어들을 다시 쉼표(,)로 연결하여 문자열로 반환
    return ", ".join(final_words)

def main():
    # 파일 경로 설정
    input_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords.csv'
    output_file = '/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv'
    
    if not os.path.exists(input_file):
        print(f"파일을 찾을 수 없습니다: {input_file}")
        return

    # CSV 파일 읽기
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    if 'keywords' not in df.columns:
        print("CSV 파일에 'keywords' 컬럼이 없습니다.")
        return

    # === 핵심 변경 부분: 띄어쓰기 제거 함수 적용 ===
    df['keywords_v2'] = df['keywords'].apply(remove_spaces)

    # 결과 확인 출력
    print("데이터 변환 완료! 변환된 샘플 5개를 확인합니다:")
    print(df[['keywords', 'keywords_v2']].head())

    # 새로운 CSV 파일로 저장
    try:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n새로운 파일이 성공적으로 저장되었습니다:\n{output_file}")
    except Exception as e:
        print(f"파일을 저장하는 중 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    main()


데이터 변환 완료! 변환된 샘플 5개를 확인합니다:
                                            keywords  \
0  럭셔리, 스테이, 리조트, 아만 다리, 호시노야 가루이자와, 스테이폴리오, 인플루언...   
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보   
2          국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색 식재료, 플레이팅, 한 끼   
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트   
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프   

                                         keywords_v2  
0  럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서,...  
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보  
2            국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색식재료, 플레이팅, 한끼  
3      성수, 합정, 홍대, 을지로, 데이트, 카페, 핫플레이스, LP, 퇴근후, 아지트  
4         오이, 소금빵, 디저트, 샌드위치, 바게트, 고리, 슬로우타운, 베이커리호프  

새로운 파일이 성공적으로 저장되었습니다:
/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv


In [9]:
# 1. 데이터가 잘 들어왔는지 먼저 확인

file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
column_name = "keywords_v2"


print(f"데이터 로드 중: {file_path}")
df = pd.read_csv(file_path)

print(f"데이터 행 개수: {len(df)}")
print(f"첫 번째 행 데이터 타입: {type(df['keywords'].iloc[0])}")
print("첫 번째 행 실제값:", repr(df['keywords_v2'].iloc[0]))


데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037
첫 번째 행 데이터 타입: <class 'str'>
첫 번째 행 실제값: '럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서, 여행, 숙박'


In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
import ast
import os


def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        # 형태 A 시도
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            # 형태 C: 대괄호 제거 후 형태 B와 동일하게 처리
            cleaned = x.strip('[]')
            # 형태 B: 쉼표 기준으로 분리하고 공백 제거
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


def main():
    # 1. 데이터 로드 및 전처리
    file_path = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    column_name = "keywords_v2"

    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path)
    print(f"데이터 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 데이터프레임에 존재하지 않습니다.")
        return

    # 데이터 타입 변환 (문자열 -> 리스트)
    df[column_name] = df[column_name].apply(safe_eval)

    # 변환 결과 샘플 확인
    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 2. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    from collections import Counter

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):  # 게시물당 1회만 카운트
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    if stopwords:
        print(f"\n[고빈도 불용어 제거] {len(stopwords)}개: {stopwords}")
    else:
        print("\n[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)")

    # 3. 네트워크 모델링 (NetworkX)
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        # 불용어 제거 후 유효 키워드만 추출
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        # 한 게시물 내 단어들로 가능한 모든 2개 조합 추출
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 4. 결과 검증 및 통계 출력
    print("-" * 50)
    print(f"네트워크 구축 완료!")
    print(f"전체 노드(키워드) 수: {G.number_of_nodes()}")
    print(f"전체 엣지(연결) 수:   {G.number_of_edges()}")
    print("-" * 50)

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 다시 확인하세요.")
        return

    # 가중치(동시 출현 빈도) 기준 상위 10개 단어 쌍 출력
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    print("\n상위 10개 단어 쌍 (동시 출현 빈도 기준):")
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    # 네트워크 기본 통계
    print("\n[네트워크 기본 통계]")
    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    print(f"  평균 연결 중심성 (Avg Degree): {avg_degree:.2f}")
    print(f"  네트워크 밀도 (Density):       {nx.density(G):.4f}")
    # 밀도 해석 안내
    density = nx.density(G)
    if density < 0.01:
        print("  → 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("  → 보통 밀도 — 정상 범위")
    else:
        print("  → 고밀도 — 불용어 추가 처리 검토 권장")


if __name__ == "__main__":
    main()

데이터 로드 중: /Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv
데이터 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)

동시 출현 네트워크(Co-occurrence Network) 생성 중...
--------------------------------------------------
네트워크 구축 완료!
전체 노드(키워드) 수: 3982
전체 엣지(연결) 수:   30508
--------------------------------------------------

상위 10개 단어 쌍 (동시 출현 빈도 기준):
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[네트워크 기본 통계]
  평균 연결 중심성 (Avg Degree): 15.32
  네트워크 밀도 (Density):       0.0038
  → 희소(Sparse) 네트워크 — 정상 범위


In [3]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 데이터 로드 및 동시 출현 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("=" * 55)
    print("[Step 1] 데이터 로드 및 네트워크 생성")
    print("=" * 55)

    # 1-1. 데이터 로드
    if not os.path.exists(file_path):
        print(f"오류: 파일을 찾을 수 없습니다 → {file_path}")
        return None

    df = pd.read_csv(file_path)
    print(f"데이터 로드 완료 | 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 존재하지 않습니다.")
        return None

    # 1-2. 키워드 컬럼 리스트 변환
    df[column_name] = df[column_name].apply(safe_eval)

    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 1-3. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    if stopwords:
        print(f"\n[고빈도 불용어 제거] {len(stopwords)}개: {stopwords}")
    else:
        print("\n[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)")

    # 1-4. 동시 출현 네트워크 생성
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 1-5. 네트워크 기본 통계 출력
    print("-" * 55)
    print(f"네트워크 구축 완료!")
    print(f"  전체 노드(키워드) 수 : {G.number_of_nodes()}")
    print(f"  전체 엣지(연결) 수   : {G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 확인하세요.")
        return None

    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    density = nx.density(G)
    print(f"  평균 연결 중심성     : {avg_degree:.2f}")
    print(f"  네트워크 밀도        : {density:.4f}", end=" ")
    if density < 0.01:
        print("→ 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("→ 보통 밀도 — 정상 범위")
    else:
        print("→ 고밀도 — 불용어 추가 처리 검토 권장")

    print("\n[상위 10개 단어 쌍 (동시 출현 빈도 기준)]")
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    return G


# ============================================================
# Step 2. 중심성 지표 계산
# ============================================================

def analyze_centrality(G, save_path="centrality_results_2025.csv"):
    print("\n" + "=" * 55)
    print("[Step 2] 중심성 지표 계산")
    print("=" * 55)

    # 2-1. 연결 중심성: 대중적 대세 트렌드
    print(" - 연결 중심성 계산 중...")
    deg_cent = nx.degree_centrality(G)

    # 2-2. 고유벡터 중심성: 파워 트렌드
    # weight='weight' 적용 — 동시 출현 빈도(친밀도)를 반영하여 계산
    print(" - 고유벡터 중심성 계산 중...")
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        print("   (⚠️ 수렴하지 않아 가중치 없이 계산합니다.)")
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 2-3. 매개 중심성: 도메인 간 융합 허브
    # weight 미사용 이유: NetworkX는 weight를 '거리(비용)'로 해석하므로
    # 동시 출현 빈도(친밀도) 기반 네트워크에서는 구조적 위치만으로 계산하는 것이 정확함
    print(" - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)")
    bet_cent = nx.betweenness_centrality(G)

    # 2-4. 결과 데이터프레임 통합
    print(" - 결과를 데이터프레임으로 정리 중...")
    centrality_df = pd.DataFrame({
        'Keyword'           : list(G.nodes()),
        'Degree (대세)'     : [deg_cent[node] for node in G.nodes()],
        'Eigenvector (파워)': [eig_cent[node] for node in G.nodes()],
        'Betweenness (융합)': [bet_cent[node] for node in G.nodes()]
    })
    centrality_df = centrality_df.sort_values(
        by='Degree (대세)', ascending=False
    ).reset_index(drop=True)

    # 2-5. 결과 저장
    centrality_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n중심성 분석 완료! 결과 저장 → '{save_path}'")

    # 2-6. 지표별 Top 5 출력
    print("\n[각 지표별 Top 5 키워드]")
    top_degree = centrality_df.sort_values(
        by='Degree (대세)', ascending=False)['Keyword'].head(5).tolist()
    top_eigen = centrality_df.sort_values(
        by='Eigenvector (파워)', ascending=False)['Keyword'].head(5).tolist()
    top_between = centrality_df.sort_values(
        by='Betweenness (융합)', ascending=False)['Keyword'].head(5).tolist()

    print(f"  대세  (Degree)      Top 5: {top_degree}")
    print(f"  파워  (Eigenvector) Top 5: {top_eigen}")
    print(f"  융합허브(Betweenness) Top 5: {top_between}")

    return centrality_df


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "centrality_results_2025.csv"

    # Step 1
    G = build_network(FILE_PATH, COLUMN_NAME)

    # Step 2
    if G is not None:
        centrality_df = analyze_centrality(G, SAVE_PATH)

[Step 1] 데이터 로드 및 네트워크 생성
데이터 로드 완료 | 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 해당 없음 (임계값: 전체 게시물의 30%)

동시 출현 네트워크(Co-occurrence Network) 생성 중...
-------------------------------------------------------
네트워크 구축 완료!
  전체 노드(키워드) 수 : 3982
  전체 엣지(연결) 수   : 30508
  평균 연결 중심성     : 15.32
  네트워크 밀도        : 0.0038 → 희소(Sparse) 네트워크 — 정상 범위

[상위 10개 단어 쌍 (동시 출현 빈도 기준)]
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[Step 2] 중심성 지표 계산
 - 연결 중심성 계산 중...
 - 고유벡터 중심성 계산 중...
 - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)
 - 결과를 데이터프레임으로 정리 중...

중심성 분석 완료! 결과 저장 → 'centrality_results_2025.csv'

[각 지표별 Top 5 키워드]
  대세  (Degree)      Top 5: ['

In [4]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    """
    다양한 형태의 키워드 데이터를 파이썬 리스트 객체로 안전하게 변환합니다.
    - 형태 A: "['디저트', '당충전']"  → ast.literal_eval 처리
    - 형태 B: "디저트, 당충전, 재충전" → split(',') 처리  ← 현재 데이터 형식
    - 형태 C: "[디저트, 당충전]"       → 대괄호 제거 후 split(',') 처리
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 데이터 로드 및 동시 출현 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("=" * 55)
    print("[Step 1] 데이터 로드 및 네트워크 생성")
    print("=" * 55)

    # 1-1. 데이터 로드
    if not os.path.exists(file_path):
        print(f"오류: 파일을 찾을 수 없습니다 → {file_path}")
        return None

    df = pd.read_csv(file_path)
    print(f"데이터 로드 완료 | 행 개수: {len(df)}")

    if column_name not in df.columns:
        print(f"오류: '{column_name}' 컬럼이 존재하지 않습니다.")
        return None

    # 1-2. 키워드 컬럼 리스트 변환
    df[column_name] = df[column_name].apply(safe_eval)

    print("\n[변환 결과 샘플]")
    for i, val in enumerate(df[column_name].head(3)):
        print(f"  [{i}] {val}")

    # 1-3. 고빈도 불용어 처리 (전체 게시물의 30% 이상 등장 키워드 제거)
    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    threshold = total_docs * 0.30
    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= threshold}

    # 매거진명, 플랫폼명 등 분석 노이즈가 되는 고유명사를 수동으로 추가
    manual_stopwords = {
        '뉴뉴',   # 매거진 플랫폼명
    }

    stopwords = high_freq_stopwords | manual_stopwords

    print(f"\n[고빈도 불용어] {len(high_freq_stopwords)}개 (임계값: 전체 게시물의 30%)")
    print(f"[수동 불용어]   {len(manual_stopwords)}개: {manual_stopwords}")
    print(f"[불용어 합계]   총 {len(stopwords)}개 제거")

    # 1-4. 동시 출현 네트워크 생성
    print("\n동시 출현 네트워크(Co-occurrence Network) 생성 중...")
    G = nx.Graph()

    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue

        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue

        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    # 1-5. 네트워크 기본 통계 출력
    print("-" * 55)
    print(f"네트워크 구축 완료!")
    print(f"  전체 노드(키워드) 수 : {G.number_of_nodes()}")
    print(f"  전체 엣지(연결) 수   : {G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        print("네트워크가 비어 있습니다. 데이터 형식을 확인하세요.")
        return None

    avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
    density = nx.density(G)
    print(f"  평균 연결 중심성     : {avg_degree:.2f}")
    print(f"  네트워크 밀도        : {density:.4f}", end=" ")
    if density < 0.01:
        print("→ 희소(Sparse) 네트워크 — 정상 범위")
    elif density < 0.1:
        print("→ 보통 밀도 — 정상 범위")
    else:
        print("→ 고밀도 — 불용어 추가 처리 검토 권장")

    print("\n[상위 10개 단어 쌍 (동시 출현 빈도 기준)]")
    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    for u, v, data in sorted_edges[:10]:
        print(f"  [{u}] - [{v}] : {data['weight']}회")

    return G


# ============================================================
# Step 2. 중심성 지표 계산
# ============================================================

def analyze_centrality(G, save_path="centrality_results_2025.csv"):
    print("\n" + "=" * 55)
    print("[Step 2] 중심성 지표 계산")
    print("=" * 55)

    # 2-1. 연결 중심성: 대중적 대세 트렌드
    print(" - 연결 중심성 계산 중...")
    deg_cent = nx.degree_centrality(G)

    # 2-2. 고유벡터 중심성: 파워 트렌드
    # weight='weight' 적용 — 동시 출현 빈도(친밀도)를 반영하여 계산
    print(" - 고유벡터 중심성 계산 중...")
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        print("   (⚠️ 수렴하지 않아 가중치 없이 계산합니다.)")
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 2-3. 매개 중심성: 도메인 간 융합 허브
    # weight 미사용 이유: NetworkX는 weight를 '거리(비용)'로 해석하므로
    # 동시 출현 빈도(친밀도) 기반 네트워크에서는 구조적 위치만으로 계산하는 것이 정확함
    print(" - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)")
    bet_cent = nx.betweenness_centrality(G)

    # 2-4. 결과 데이터프레임 통합
    print(" - 결과를 데이터프레임으로 정리 중...")
    centrality_df = pd.DataFrame({
        'Keyword'           : list(G.nodes()),
        'Degree (대세)'     : [deg_cent[node] for node in G.nodes()],
        'Eigenvector (파워)': [eig_cent[node] for node in G.nodes()],
        'Betweenness (융합)': [bet_cent[node] for node in G.nodes()]
    })
    centrality_df = centrality_df.sort_values(
        by='Degree (대세)', ascending=False
    ).reset_index(drop=True)

    # 2-5. 결과 저장
    centrality_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n중심성 분석 완료! 결과 저장 → '{save_path}'")

    # 2-6. 지표별 Top 5 출력
    print("\n[각 지표별 Top 5 키워드]")
    top_degree = centrality_df.sort_values(
        by='Degree (대세)', ascending=False)['Keyword'].head(5).tolist()
    top_eigen = centrality_df.sort_values(
        by='Eigenvector (파워)', ascending=False)['Keyword'].head(5).tolist()
    top_between = centrality_df.sort_values(
        by='Betweenness (융합)', ascending=False)['Keyword'].head(5).tolist()

    print(f"  대세  (Degree)      Top 5: {top_degree}")
    print(f"  파워  (Eigenvector) Top 5: {top_eigen}")
    print(f"  융합허브(Betweenness) Top 5: {top_between}")

    return centrality_df


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "centrality_results_2025.csv"

    # Step 1
    G = build_network(FILE_PATH, COLUMN_NAME)

    # Step 2
    if G is not None:
        centrality_df = analyze_centrality(G, SAVE_PATH)

[Step 1] 데이터 로드 및 네트워크 생성
데이터 로드 완료 | 행 개수: 1037

[변환 결과 샘플]
  [0] ['럭셔리', '스테이', '리조트', '아만다리', '호시노야가루이자와', '스테이폴리오', '인플루언서', '여행', '숙박']
  [1] ['롯데리아', '나폴리', '감자연구소', '디저트', '쥐포튀김', '떼리앙', '캐릭터', '띠부씰', '콜라보']
  [2] ['국밥', '뉴웨이브', 'MZ세대', '힙함', '감성', '이색식재료', '플레이팅', '한끼']

[고빈도 불용어] 0개 (임계값: 전체 게시물의 30%)
[수동 불용어]   1개: {'뉴뉴'}
[불용어 합계]   총 1개 제거

동시 출현 네트워크(Co-occurrence Network) 생성 중...
-------------------------------------------------------
네트워크 구축 완료!
  전체 노드(키워드) 수 : 3981
  전체 엣지(연결) 수   : 30194
  평균 연결 중심성     : 15.17
  네트워크 밀도        : 0.0038 → 희소(Sparse) 네트워크 — 정상 범위

[상위 10개 단어 쌍 (동시 출현 빈도 기준)]
  [카페] - [맛집] : 70회
  [디저트] - [카페] : 59회
  [카페] - [브런치] : 43회
  [여행] - [카페] : 41회
  [카페] - [커피] : 40회
  [감성] - [카페] : 31회
  [카페] - [F&B] : 28회
  [카페] - [빈티지] : 28회
  [카페] - [힐링] : 27회
  [디저트] - [커피] : 26회

[Step 2] 중심성 지표 계산
 - 연결 중심성 계산 중...
 - 고유벡터 중심성 계산 중...
 - 매개 중심성 계산 중... (⏳ 1~3분 소요될 수 있습니다)
 - 결과를 데이터프레임으로 정리 중...

중심성 분석 완료! 결과 저장 → 'centrality_results_2025.csv'

[각 지표별 To

In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os

# Louvain 알고리즘 (python-louvain 패키지)
try:
    import community as community_louvain
except ImportError:
    print("python-louvain 패키지가 없습니다. 설치 중...")
    os.system("pip install python-louvain")
    import community as community_louvain

# 인터랙티브 시각화 (pyvis 패키지)
try:
    from pyvis.network import Network
except ImportError:
    print("pyvis 패키지가 없습니다. 설치 중...")
    os.system("pip install pyvis")
    from pyvis.network import Network


# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= total_docs * 0.30}
    manual_stopwords = {
        '뉴뉴',  # 매거진 플랫폼명
    }
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
    return G


# ============================================================
# Step 4. Louvain 메가트렌드 군집화
# ============================================================

def detect_communities(G):
    print("\n[Step 4] Louvain 커뮤니티 탐지 중...")
    partition = community_louvain.best_partition(G, weight='weight', random_state=42)

    community_sizes = Counter(partition.values())
    n_communities = len(community_sizes)
    print(f"  발견된 군집 수: {n_communities}개")

    deg_cent = nx.degree_centrality(G)
    print("\n[군집별 Top 5 키워드 (연결 중심성 기준)]")
    print("-" * 55)
    for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        top5 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:5]
        print(f"  군집 {comm_id:2d} ({len(members):4d}개): {top5}")
    print("-" * 55)

    return partition


# ============================================================
# 시각화: 전체 네트워크 인터랙티브 HTML
# ============================================================

# 군집 ID → 색상 팔레트 (20가지)
PALETTE = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
    "#8BC34A", "#FF5722", "#607D8B", "#795548", "#FF9800",
    "#673AB7", "#009688", "#F44336", "#2196F3", "#4CAF50",
]


def build_visualization(G, partition, save_path="trend_network_full.html"):
    print("\n[시각화] 전체 네트워크 HTML 생성 중...")

    # 연결 중심성 (대세) → 노드 크기
    deg_cent = nx.degree_centrality(G)

    # 고유벡터 중심성 (파워) → 노드 테두리 두께
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    # 고유벡터 중심성 정규화 → 테두리 두께 1~10px
    eig_values = list(eig_cent.values())
    eig_min, eig_max = min(eig_values), max(eig_values)

    def eig_to_border(val):
        normalized = (val - eig_min) / (eig_max - eig_min + 1e-9)
        return 1 + normalized * 9  # 1px ~ 10px

    community_sizes = Counter(partition.values())

    # 군집 ID를 크기 순으로 정렬하여 색상 인덱스 부여
    comm_rank = {
        comm_id: rank
        for rank, (comm_id, _) in enumerate(
            sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)
        )
    }

    # pyvis 네트워크 설정
    net = Network(
        height="950px",
        width="100%",
        bgcolor="#1a1a2e",
        font_color="#ffffff",
        notebook=False
    )

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.005,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "maxVelocity": 50,
        "minVelocity": 0.1,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
          "enabled": true,
          "iterations": 200
        }
      },
      "nodes": {
        "shape": "dot",
        "font": {
          "size": 12,
          "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"
        },
        "borderWidthSelected": 4
      },
      "edges": {
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }
    """)

    # 노드 추가
    for node in G.nodes():
        comm_id      = partition[node]
        color_idx    = comm_rank.get(comm_id, 0) % len(PALETTE)
        color        = PALETTE[color_idx]
        degree       = deg_cent[node]
        border_width = eig_to_border(eig_cent[node])

        # 연결 중심성 → 노드 크기 (5~40px)
        size = min(5 + degree * 120, 40)

        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"군집: {comm_id}\n"
                f"─────────────────\n"
                f"크기   → 대세  (연결 중심성):     {deg_cent[node]:.4f}\n"
                f"테두리 → 파워  (고유벡터 중심성): {eig_cent[node]:.4f}\n"
                f"─────────────────\n"
                f"군집 크기: {community_sizes[comm_id]}개"
            )
        )

    # 엣지 추가 (가중치 낮은 엣지는 흐리게)
    max_weight = max(d['weight'] for _, _, d in G.edges(data=True))
    for u, v, data in G.edges(data=True):
        w       = data['weight']
        opacity = 0.05 + 0.4 * (w / max_weight)
        width   = 0.5 + 2.5 * (w / max_weight)
        net.add_edge(
            u, v,
            value=w,
            width=width,
            color={"opacity": opacity},
            title=f"{u} ↔ {v}\n동시 출현: {w}회"
        )

    net.save_graph(save_path)
    print(f"  저장 완료 → '{save_path}'")
    print("  브라우저에서 파일을 열어 확인하세요.")
    print("  (노드 드래그 · 확대/축소 · 호버 시 지표 확인 가능)")


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "/Users/hajiyoon/workspace/trend_network_full.html"3

    G         = build_network(FILE_PATH, COLUMN_NAME)
    partition = detect_communities(G)
    build_visualization(G, partition, SAVE_PATH)

[Step 1] 네트워크 생성 중...
  불용어 제거: 총 1개
  노드: 3981개 | 엣지: 30194개

[Step 4] Louvain 커뮤니티 탐지 중...
  발견된 군집 수: 34개

[군집별 Top 5 키워드 (연결 중심성 기준)]
-------------------------------------------------------
  군집  1 ( 832개): ['카페', '디저트', '맛집', '커피', '브런치']
  군집  6 ( 451개): ['말차', 'CU', '딸기', '제주', '스타벅스']
  군집  7 ( 394개): ['F&B', '성수', '와인', '연말', '크리스마스']
  군집  8 ( 250개): ['케이크', '겨울', '음식', '가을', '선물']
  군집 10 ( 227개): ['여름', '제철', '달콤', '빙수', '상큼']
  군집  2 ( 168개): ['팝업', '콜라보', '타코', '플리마켓', '홍콩']
  군집  4 ( 161개): ['레트로', '저녁', '점심', '버거', '만두']
  군집 22 ( 149개): ['안주', '한식', '퓨전', '맥주', '연희동']
  군집 12 ( 146개): ['떡볶이', '오뎅', '하이볼', '가성비', '야식']
  군집 18 ( 132개): ['쇼핑', '도쿄', '라멘', '휴식', '족발']
  군집 17 ( 128개): ['일본', '바삭', '일식', '직장인', '키링']
  군집  3 ( 118개): ['빵', '베이글', '빵지순례', '빵순이', '초여름']
  군집 11 ( 115개): ['간식', '추억', '공간', '킷사텐', '제니']
  군집 20 ( 102개): ['건강', '비건', '김밥', '아침', '다이어트']
  군집 19 (  82개): ['흑백요리사', '시즌', '밥친구', '셰프', '핫플']
  군집  5 (  77개): ['당충전', '한남동', '미식', '체험', '독일']
  군집  0

In [8]:
# 기존 코드 실행 후 G, partition이 있는 상태에서 아래 추가 실행

import pandas as pd
import networkx as nx
from collections import Counter

deg_cent = nx.degree_centrality(G)
community_sizes = Counter(partition.values())

rows = []
for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
    members = [node for node, c in partition.items() if c == comm_id]
    top10 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:10]
    rows.append({
        '군집ID':    comm_id,
        '키워드수':  community_sizes[comm_id],
        'Top10키워드': ', '.join(top10),
        '메가트렌드명': ''  # 나중에 직접 채울 칸
    })

result_df = pd.DataFrame(rows)
result_df.to_csv('/Users/hajiyoon/workspace/megatrend_naming.csv',
                 index=False, encoding='utf-8-sig')
print(result_df[['군집ID','키워드수','Top10키워드']].to_string())

    군집ID  키워드수                                                     Top10키워드
0      1   832                   카페, 디저트, 맛집, 커피, 브런치, 데이트, 여행, 빈티지, 감성, 힐링
1      6   451               말차, CU, 딸기, 제주, 스타벅스, 아이스크림, 피스타치오, 굿즈, 망고, 미국
2      7   394              F&B, 성수, 와인, 연말, 크리스마스, 샌드위치, 파스타, 이자카야, 신상, 피자
3      8   250                   케이크, 겨울, 음식, 가을, 선물, 브랜드, 취향, 친구, 무화과, 디자인
4     10   227                     여름, 제철, 달콤, 빙수, 상큼, 과일, 파르페, 음료, 성심당, 셀럽
5      2   168               팝업, 콜라보, 타코, 플리마켓, 홍콩, 캐릭터, 성수동, 막걸리, 큐레이션, 파리
6      4   161                  레트로, 저녁, 점심, 버거, 만두, 아메리칸, 국밥, 닭갈비, 다방, 사랑방
7     22   149                     안주, 한식, 퓨전, 맥주, 연희동, 음악, 혼술, 햄버거, 맛, 위스키
8     12   146                  떡볶이, 오뎅, 하이볼, 가성비, 야식, 대구, 야장, 시원함, 플라워, 노포
9     18   132                      쇼핑, 도쿄, 라멘, 휴식, 족발, 메뉴, 편집샵, 루틴, 전시, 스시
10    17   128                  일본, 바삭, 일식, 직장인, 키링, 사시미, 감칠맛, 오리지널, 식감, 카츠
11     3   118              빵, 베이글, 빵지순례, 빵순이, 초여름, 슈톨렌, 카라멜, 빵덕후, 호두과자, 천안
12    11   1

In [9]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os

try:
    import community as community_louvain
except ImportError:
    os.system("pip install python-louvain")
    import community as community_louvain

try:
    from pyvis.network import Network
except ImportError:
    os.system("pip install pyvis")
    from pyvis.network import Network


# ============================================================
# 메가트렌드 명명 딕셔너리 (군집ID → 메가트렌드명)
# ============================================================

MEGATREND_NAMES = {
    1:  "감성 카페 라이프",
    6:  "시즌 한정 디저트",
    7:  "핫플레이스 다이닝",
    8:  "선물형 디저트 브랜딩",
    10: "제철 과일 디저트",
    2:  "팝업·콜라보 경험",
    4:  "레트로 로컬 푸드",
    22: "혼술·홈술 문화",
    12: "가성비 야식·노포",
    18: "라이프스타일 쇼핑",
    17: "일식 열풍",
    3:  "빵지순례 문화",
    11: "추억·향수 간식",
    20: "건강식·비건",
    19: "미디어 푸드 콘텐츠",
    5:  "미식 체험",
    0:  "먹방·여행 콘텐츠",
    9:  "골목 술자리 문화",
    23: "수험생·이벤트 특수",
    13: "비주얼 데이트 플레이스",
    14: "프리미엄 파인다이닝",
    15: "도심 SNS 스팟",
    30: "연말연시 이벤트",
    21: "프리미엄 버거 재료",
    28: "시즌 한정 음료",
    27: "럭셔리 브랜드 F&B",
    24: "패션·다이닝 크로스오버",
    26: "로컬 여행 맛집",
    16: "빈티지 라이프스타일",
    31: "인플루언서 맛집",
    25: "간편결제 라이프",
    29: "대학가 음식 문화",
    32: "엔터테인먼트 스팟",
    33: "배달·푸드테크",
}

# ============================================================
# 유틸리티 함수
# ============================================================

def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip('[]')
            return [kw.strip() for kw in cleaned.split(',') if kw.strip()]
        return []


# ============================================================
# Step 1. 네트워크 생성
# ============================================================

def build_network(file_path, column_name):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= total_docs * 0.30}
    manual_stopwords = {'뉴뉴'}
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")
    return G


# ============================================================
# Step 4. Louvain 메가트렌드 군집화
# ============================================================

def detect_communities(G):
    print("\n[Step 4] Louvain 커뮤니티 탐지 중...")
    partition = community_louvain.best_partition(G, weight='weight', random_state=42)
    community_sizes = Counter(partition.values())
    print(f"  발견된 군집 수: {len(community_sizes)}개")
    return partition


# ============================================================
# 시각화
# ============================================================

PALETTE = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
    "#8BC34A", "#FF5722", "#607D8B", "#795548", "#FF9800",
    "#673AB7", "#009688", "#F44336", "#2196F3", "#4CAF50",
    "#CDDC39", "#FF5252", "#40C4FF", "#69F0AE", "#FFD740",
    "#E040FB", "#00E5FF", "#76FF03", "#FF6D00", "#651FFF",
    "#F50057", "#00B0FF", "#C6FF00", "#FFAB40", "#D500F9",
]


def build_visualization(G, partition, save_path="trend_network_full.html"):
    print("\n[시각화] HTML 생성 중...")

    deg_cent = nx.degree_centrality(G)

    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    eig_values = list(eig_cent.values())
    eig_min, eig_max = min(eig_values), max(eig_values)

    def eig_to_border(val):
        normalized = (val - eig_min) / (eig_max - eig_min + 1e-9)
        return 1 + normalized * 9

    community_sizes = Counter(partition.values())

    comm_rank = {
        comm_id: rank
        for rank, (comm_id, _) in enumerate(
            sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)
        )
    }

    net = Network(
        height="950px",
        width="100%",
        bgcolor="#1a1a2e",
        font_color="#ffffff",
        notebook=False
    )

    # 안정화 후 물리 엔진 OFF + 클릭 시 연결 노드만 표시
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.005,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "maxVelocity": 50,
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {
          "enabled": true,
          "iterations": 300,
          "updateInterval": 50,
          "fit": true
        }
      },
      "nodes": {
        "shape": "dot",
        "font": {
          "size": 12,
          "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"
        },
        "borderWidthSelected": 4
      },
      "edges": {
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true,
        "selectConnectedEdges": true
      }
    }
    """)

    for node in G.nodes():
        comm_id      = partition[node]
        megatrend    = MEGATREND_NAMES.get(comm_id, f"군집 {comm_id}")
        color_idx    = comm_rank.get(comm_id, 0) % len(PALETTE)
        color        = PALETTE[color_idx]
        degree       = deg_cent[node]
        border_width = eig_to_border(eig_cent[node])
        size         = min(5 + degree * 120, 40)

        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"메가트렌드: {megatrend}\n"
                f"─────────────────\n"
                f"크기   → 대세  (연결 중심성):     {deg_cent[node]:.4f}\n"
                f"테두리 → 파워  (고유벡터 중심성): {eig_cent[node]:.4f}\n"
                f"─────────────────\n"
                f"군집 크기: {community_sizes[comm_id]}개"
            ),
            group=str(comm_id)
        )

    max_weight = max(d['weight'] for _, _, d in G.edges(data=True))
    for u, v, data in G.edges(data=True):
        w       = data['weight']
        opacity = 0.05 + 0.4 * (w / max_weight)
        width   = 0.5 + 2.5 * (w / max_weight)
        net.add_edge(
            u, v,
            value=w,
            width=width,
            color={"opacity": opacity},
            title=f"{u} ↔ {v}\n동시 출현: {w}회"
        )

    # HTML 저장 후 JS 주입
    net.save_graph(save_path)

    with open(save_path, 'r', encoding='utf-8') as f:
        html = f.read()

    # 1) 안정화 완료 후 물리 엔진 OFF (꿈틀거림 제거)
    # 2) 노드 클릭 시 해당 노드 + 연결된 노드/엣지만 표시, 나머지 희미하게
    custom_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {

  // 안정화 완료 후 물리 엔진 OFF
  network.once("stabilized", function() {
    network.setOptions({ physics: { enabled: false } });
  });

  // 노드 클릭 시 연결 노드만 강조
  network.on("click", function(params) {
    if (params.nodes.length > 0) {
      var selectedNode = params.nodes[0];
      var connectedNodes = network.getConnectedNodes(selectedNode);
      var connectedEdges = network.getConnectedEdges(selectedNode);

      var allNodes = nodes.get();
      var allEdges = edges.get();

      // 연결되지 않은 노드는 희미하게
      var updateNodes = allNodes.map(function(n) {
        if (n.id === selectedNode || connectedNodes.indexOf(n.id) !== -1) {
          return { id: n.id, opacity: 1.0 };
        } else {
          return { id: n.id, opacity: 0.05 };
        }
      });

      // 연결되지 않은 엣지는 희미하게
      var updateEdges = allEdges.map(function(e) {
        if (connectedEdges.indexOf(e.id) !== -1) {
          return { id: e.id, color: { opacity: 0.9 } };
        } else {
          return { id: e.id, color: { opacity: 0.02 } };
        }
      });

      nodes.update(updateNodes);
      edges.update(updateEdges);

    } else {
      // 빈 곳 클릭 시 원래대로 복원
      var allNodes = nodes.get();
      var allEdges = edges.get();

      var resetNodes = allNodes.map(function(n) {
        return { id: n.id, opacity: 1.0 };
      });
      var resetEdges = allEdges.map(function(e) {
        return { id: e.id, color: { opacity: null } };
      });

      nodes.update(resetNodes);
      edges.update(resetEdges);
    }
  });
});
</script>
"""

    # </body> 직전에 JS 삽입
    html = html.replace('</body>', custom_js + '\n</body>')

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"  저장 완료 → '{save_path}'")
    print("  [사용법]")
    print("  - 노드 클릭: 연결된 노드만 강조, 나머지 희미해짐")
    print("  - 빈 곳 클릭: 전체 복원")
    print("  - 호버: 키워드·메가트렌드명·지표 확인")
    print("  - 네트워크는 안정화 후 자동으로 고정됩니다")


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":
    FILE_PATH   = "/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv"
    COLUMN_NAME = "keywords_v2"
    SAVE_PATH   = "/Users/hajiyoon/workspace/trend_network_full.html"

    G         = build_network(FILE_PATH, COLUMN_NAME)
    partition = detect_communities(G)
    build_visualization(G, partition, SAVE_PATH)

[Step 1] 네트워크 생성 중...
  불용어 제거: 총 1개
  노드: 3981개 | 엣지: 30194개

[Step 4] Louvain 커뮤니티 탐지 중...
  발견된 군집 수: 34개

[시각화] HTML 생성 중...
  저장 완료 → '/Users/hajiyoon/workspace/trend_network_full.html'
  [사용법]
  - 노드 클릭: 연결된 노드만 강조, 나머지 희미해짐
  - 빈 곳 클릭: 전체 복원
  - 호버: 키워드·메가트렌드명·지표 확인
  - 네트워크는 안정화 후 자동으로 고정됩니다


In [11]:
import pandas as pd

df = pd.read_csv("/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv")

# 전체 컬럼 확인
print(df.columns.tolist())

print(df['date'].head(10))
print("\n타입:", df['date'].dtype)

['post_id', 'date', 'timestamp', 'title', 'body', 'likes', 'comments', 'url', '광고 미포함', 'keywords', 'keywords_v2']
0    2025-04-21
1    2025-04-21
2    2025-04-21
3    2025-04-22
4    2025-04-22
5    2025-04-22
6    2025-04-23
7    2025-04-23
8    2025-04-24
9    2025-04-24
Name: date, dtype: str

타입: str


In [12]:
import pandas as pd

df = pd.read_csv("/Users/hajiyoon/workspace/data/processed/knewnew_without_ad_with_keywords_v2.csv")

df['date'] = pd.to_datetime(df['date'])

print("시작일:", df['date'].min())
print("종료일:", df['date'].max())
print("총 기간:", (df['date'].max() - df['date'].min()).days, "일")
print("\n월별 게시물 수:")
print(df.groupby(df['date'].dt.to_period('M')).size().to_string())

시작일: 2025-04-21 00:00:00
종료일: 2025-12-31 00:00:00
총 기간: 254 일

월별 게시물 수:
date
2025-04     34
2025-05    104
2025-06    106
2025-07    130
2025-08    125
2025-09    115
2025-10    122
2025-11    142
2025-12    159
Freq: M
